# 3일차 실습 1. API Key 발급 및 기본 호출 테스트

## 실습 목표

이번 실습에서는 `requests`를 사용해 교육 환경에서 제공되는 생성형 AI API(MLAPI)에 한 번의 호출을 보내고, 정상 응답을 받는 과정을 익힙니다.

이번 실습에서 다루는 내용은 다음과 같습니다.

- MLAPI 접속 URL과 API Key 준비
- `.env` 파일로 필요한 설정값 분리
- 환경변수 또는 임시 입력으로 API Key 불러오기
- Chat Completions 형태의 `payload` 구성
- `requests.post()`로 기본 요청 보내기
- 응답 상태 코드와 응답 본문 확인
- 응답 JSON에서 답변 텍스트 추출하기


## 1. 라이브러리 불러오기

이번 실습에서 사용할 표준 라이브러리와 외부 패키지를 한 번에 불러옵니다.

In [ ]:
import os
import json
import requests
from getpass import getpass
from dotenv import load_dotenv

## 2. 접속 정보 구조 살펴보기

이번 실습에서는 다음 형태의 요청 코드를 사용합니다.

```python
response = requests.post(endpoint, headers=headers, json=payload)
```

API 호출에는 보통 다음 정보가 필요합니다.

```
MLAPI_BASE_URL → 요청을 보낼 API 베이스 주소 (끝이 /v1)
MLAPI_API_KEY  → 인증에 사용할 API Key
MLAPI_MODEL    → 사용할 모델 이름 (프로바이더 접두사 포함)
```

이번 실습은 `gpt-4o-mini` 기준 MLAPI 베이스 URL을 사용합니다.

```text
https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1
```

MLAPI는 OpenAI Chat Completions와 동일한 스펙을 따르므로 실제 호출 시에는 베이스 URL 뒤에 `/chat/completions`를 붙여 사용합니다. 모델명은 `openai/gpt-4o-mini` 처럼 **프로바이더 접두사가 포함된 형태**로 지정합니다.

아래 셀은 실제 비밀값이 아닌 예시 `.env` 파일을 만듭니다.

In [ ]:
%%writefile .env.example
MLAPI_BASE_URL=https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1
MLAPI_API_KEY=여기에_MLAPI_KEY를_입력하세요
MLAPI_MODEL=openai/gpt-4o-mini

## 3. 환경변수 불러오기

`.env` 파일이 있는 경우 값을 불러옵니다.

API Key가 환경변수에 없으면 노트북 실행 중 임시로 입력할 수 있습니다. 입력한 값은 코드 셀 출력에 표시되지 않습니다.

In [ ]:
load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

print("MLAPI Base URL:", base_url)
print("모델명:", model_name)
print("API Key 입력 여부:", bool(api_key))

## 4. 기본 payload 구성하기

MLAPI에 보낼 요청 Body를 구성합니다.

이번 실습에서는 Chat Completions 형태에 맞춰 `model`, `messages`, `temperature`를 사용합니다.

In [ ]:
payload = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "너는 FastAPI와 백엔드 API를 쉽게 설명하는 교육 조교야."},
        {"role": "user",   "content": "FastAPI를 한 문장으로 설명해줘."}
    ],
    "temperature": 0.2
}

print(json.dumps(payload, ensure_ascii=False, indent=2))

## 5. 헤더 구성하기

MLAPI는 `Authorization: Bearer <API_KEY>` 헤더로 인증합니다.

Key 값 자체는 노출되지 않도록, 출력용으로는 마스킹한 사본을 만들어 확인합니다.

In [ ]:
headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}"
}

safe_headers = {**headers, "Authorization": "Bearer ***"}
print(json.dumps(safe_headers, ensure_ascii=False, indent=2))

## 6. requests로 기본 호출 실행하기

`requests.post()`를 사용해 MLAPI에 요청을 보냅니다.

엔드포인트는 베이스 URL 뒤에 `/chat/completions`를 붙여 만듭니다. 네트워크 상태, API Key, 모델명, payload 구조에 따라 오류가 발생할 수 있습니다.

In [ ]:
endpoint = base_url.rstrip("/") + "/chat/completions"

try:
    response = requests.post(endpoint, json=payload, headers=headers, timeout=60)

    print("호출 완료")
    print("Status Code:", response.status_code)
    print("Response Text:")
    print(response.text[:1000])

except Exception as e:
    response = None
    print("호출 실패")
    print("오류 타입:", type(e).__name__)
    print("오류 메시지:", str(e)[:500])

## 7. 응답 JSON 구조 확인하기

응답이 JSON 형태라면 Python dict로 변환해 구조를 확인합니다.

In [ ]:
if response is not None:
    try:
        response_json = response.json()

        print("응답 최상위 Key")
        print(list(response_json.keys()))

        print("\n응답 JSON 일부")
        print(json.dumps(response_json, ensure_ascii=False, indent=2)[:1500])

    except Exception as e:
        response_json = None
        print("JSON 변환 실패")
        print("오류 타입:", type(e).__name__)
        print("오류 메시지:", str(e)[:500])
else:
    response_json = None
    print("response가 없어 JSON 구조를 확인하지 않습니다.")

## 8. 답변 텍스트 추출하기

OpenAI Chat Completions 형태의 응답이라면 답변 텍스트는 보통 다음 위치에 있습니다.

```
choices[0].message.content
```

MLAPI 응답 구조가 다를 수 있으므로, 먼저 실제 응답 JSON 구조를 확인한 뒤 필요한 값을 추출합니다.

In [ ]:
if response_json is not None:
    try:
        answer = response_json["choices"][0]["message"]["content"].strip()
        usage  = response_json.get("usage", {})

        print("LLM 답변")
        print("-" * 60)
        print(answer)
        print("-" * 60)
        print("사용 토큰:", usage)
    except Exception as e:
        print("답변 추출 실패")
        print("오류 타입:", type(e).__name__)
        print("오류 메시지:", str(e)[:500])
else:
    print("response_json이 없어 답변을 추출하지 않습니다.")

## 9. 실패했을 때 자주 보는 신호 (참고)

| Status | 의미 | 가장 흔한 원인 |
|---|---|---|
| 401 | 인증 실패 | `MLAPI_API_KEY`가 비었거나 잘못됨 |
| 404 | 경로 없음 | 엔드포인트에 `/chat/completions`가 빠짐 |
| 400 | 요청 오류 | `model` 이름이 잘못되었거나 필수 필드 누락 |
| 429 | 속도 제한 | 잠시 후 재시도 |

위 표는 참고용입니다. 실제 에러 처리·재시도 로직은 다음 차시(*LLM API 에러 처리 및 안정성*)에서 다룹니다.

## 실습 정리

이번 실습에서 수행한 내용은 다음과 같습니다.

- MLAPI 베이스 URL과 API Key 구조를 확인했습니다.
- `.env.example` 구조를 살펴보고 환경변수로 설정값을 분리했습니다.
- Chat Completions 형태의 payload를 구성했습니다.
- `requests.post()`로 생성형 AI API를 호출했습니다.
- 응답 상태 코드와 응답 본문을 확인했습니다.
- 응답 JSON에서 답변 텍스트를 추출했습니다.